# Traditional ML Baseline for Restaurant ABSA

This notebook trains and evaluates the traditional machine learning baseline for the restaurant Aspect-Based Sentiment Analysis project.

It uses the processed dataset created by `backends/data/preprocess_dataset.ipynb`, not the old raw SemEval CSV path.

## Step 1: Import Libraries and Define Paths

The notebook is stored in `backends/traditional_ml/`. The processed dataset is stored in `backends/data/processed/`. The trained model and evaluation report are saved back into this folder.

In [1]:
from pathlib import Path
import json

import joblib
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

TRADITIONAL_ML_DIR = Path.cwd()
BACKENDS_DIR = TRADITIONAL_ML_DIR.parent
DATASET_PATH = BACKENDS_DIR / "data" / "processed" / "restaurant_absa_4_aspects.csv"
MODEL_PATH = TRADITIONAL_ML_DIR / "model.joblib"
REPORT_PATH = TRADITIONAL_ML_DIR / "training_report.json"

ASPECTS = [
    "Food",
    "Service",
    "Price",
    "Eating Environment / Ambiance",
]
LABELS = ["Positive", "Negative", "Unknown"]

print("Notebook directory:", TRADITIONAL_ML_DIR)
print("Dataset path:", DATASET_PATH)
print("Model output:", MODEL_PATH)
print("Report output:", REPORT_PATH)

Notebook directory: /Users/kafe/Desktop/NLP_Final Project/absa_v01/backends/traditional_ml
Dataset path: /Users/kafe/Desktop/NLP_Final Project/absa_v01/backends/data/processed/restaurant_absa_4_aspects.csv
Model output: /Users/kafe/Desktop/NLP_Final Project/absa_v01/backends/traditional_ml/model.joblib
Report output: /Users/kafe/Desktop/NLP_Final Project/absa_v01/backends/traditional_ml/training_report.json


## Step 2: Load and Validate the Processed Dataset

The processed wide dataset has one row per review and one label column for each fixed project aspect. Every missing aspect has already been filled with `Unknown`.

In [2]:
df = pd.read_csv(DATASET_PATH, dtype={"id": str})

expected_columns = ["id", "review", *ASPECTS]
assert list(df.columns) == expected_columns, f"Unexpected columns: {list(df.columns)}"
assert df["review"].notna().all(), "Some reviews are missing."
assert df["review"].str.len().gt(0).all(), "Some reviews are empty."

for aspect in ASPECTS:
    bad_labels = sorted(set(df[aspect]) - set(LABELS))
    assert not bad_labels, f"{aspect} has invalid labels: {bad_labels}"

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
print("Label distribution across all aspect columns:")
print(df[ASPECTS].stack().value_counts())

df.head()

Dataset shape: (3041, 6)
Columns: ['id', 'review', 'Food', 'Service', 'Price', 'Eating Environment / Ambiance']
Label distribution across all aspect columns:
Unknown     9891
Positive    1633
Negative     640
Name: count, dtype: int64


,id,review,Food,Service,Price,Eating Environment / Ambiance
0,3121,But the staff was so horrible to us.,Unknown,Negative,Unknown,Unknown
1,2777,"To be completely fair, the only redeeming fact...",Positive,Unknown,Unknown,Unknown
2,1634,"The food is uniformly exceptional, with a very...",Positive,Unknown,Unknown,Unknown
3,2534,Where Gabriela personaly greets you and recomm...,Unknown,Positive,Unknown,Unknown
4,583,"For those that go once and don't enjoy it, all...",Unknown,Unknown,Unknown,Unknown


## Step 3: Split the Dataset

We use an 80/20 train-test split. The same train/test reviews are reused for all four aspect classifiers so the evaluation is consistent.

In [3]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

print("Training rows:", len(train_df))
print("Testing rows:", len(test_df))

Training rows: 2432
Testing rows: 609


## Step 4: Build the Traditional ML Pipeline

The model uses TF-IDF text features and Logistic Regression.

- TF-IDF converts each review into numeric word/ngram features.
- Logistic Regression is a simple supervised classifier that is easy to explain.
- `class_weight="balanced"` helps because most aspect labels are `Unknown`.

In [4]:
def build_pipeline() -> Pipeline:
    return Pipeline(
        steps=[
            (
                "tfidf",
                TfidfVectorizer(
                    lowercase=True,
                    ngram_range=(1, 2),
                    min_df=2,
                    max_features=20000,
                ),
            ),
            (
                "classifier",
                LogisticRegression(
                    max_iter=1000,
                    class_weight="balanced",
                    solver="liblinear",
                    random_state=42,
                ),
            ),
        ]
    )

## Step 5: Train One Classifier per Aspect

The project always outputs four aspects. For a simple baseline, we train four separate classifiers: one each for Food, Service, Price, and Eating Environment / Ambiance.

In [5]:
models = {}

for aspect in ASPECTS:
    pipeline = build_pipeline()
    pipeline.fit(train_df["review"], train_df[aspect])
    models[aspect] = pipeline
    print(f"Trained classifier for: {aspect}")

Trained classifier for: Food
Trained classifier for: Service
Trained classifier for: Price
Trained classifier for: Eating Environment / Ambiance


## Step 6: Evaluate the Models

The report includes accuracy, macro F1, weighted F1, and the per-label classification report. Macro F1 is especially useful because it treats `Positive`, `Negative`, and `Unknown` more evenly.

In [6]:
report = {
    "dataset": str(DATASET_PATH),
    "rows": int(len(df)),
    "train_rows": int(len(train_df)),
    "test_rows": int(len(test_df)),
    "aspects": ASPECTS,
    "labels": LABELS,
    "metrics": {},
}

for aspect in ASPECTS:
    y_true = test_df[aspect]
    y_pred = models[aspect].predict(test_df["review"])

    report["metrics"][aspect] = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, labels=LABELS, average="macro")),
        "weighted_f1": float(f1_score(y_true, y_pred, labels=LABELS, average="weighted")),
        "classification_report": classification_report(
            y_true,
            y_pred,
            labels=LABELS,
            output_dict=True,
            zero_division=0,
        ),
        "confusion_matrix": confusion_matrix(y_true, y_pred, labels=LABELS).tolist(),
    }

summary_rows = []
for aspect, metrics in report["metrics"].items():
    summary_rows.append({
        "aspect": aspect,
        "accuracy": metrics["accuracy"],
        "macro_f1": metrics["macro_f1"],
        "weighted_f1": metrics["weighted_f1"],
    })

summary_df = pd.DataFrame(summary_rows)
summary_df

,aspect,accuracy,macro_f1,weighted_f1
0,Food,0.821018,0.668644,0.810514
1,Service,0.855501,0.574808,0.831019
2,Price,0.929392,0.590400,0.909513
3,Eating Environment / Ambiance,0.894910,0.512967,0.872937


## Step 7: Inspect Per-Aspect Reports

These text reports are useful for the final presentation because they show precision, recall, and F1-score for every label.

In [7]:
for aspect in ASPECTS:
    print("=" * 80)
    print(aspect)
    print("=" * 80)
    y_true = test_df[aspect]
    y_pred = models[aspect].predict(test_df["review"])
    print(classification_report(y_true, y_pred, labels=LABELS, zero_division=0))

Food
              precision    recall  f1-score   support

    Positive       0.74      0.67      0.70       151
    Negative       0.62      0.31      0.41        42
     Unknown       0.85      0.93      0.89       416

    accuracy                           0.82       609
   macro avg       0.74      0.64      0.67       609
weighted avg       0.81      0.82      0.81       609

Service
              precision    recall  f1-score   support

    Positive       0.59      0.50      0.54        58
    Negative       0.56      0.16      0.25        55
     Unknown       0.89      0.97      0.93       496

    accuracy                           0.86       609
   macro avg       0.68      0.55      0.57       609
weighted avg       0.83      0.86      0.83       609

Price
              precision    recall  f1-score   support

    Positive       0.87      0.43      0.58        30
    Negative       0.80      0.13      0.23        30
     Unknown       0.93      1.00      0.96       549

 

## Step 8: Save the Model and Training Report

The saved `model.joblib` is used by `predict.py`, and `training_report.json` keeps the evaluation results for documentation.

In [8]:
artifact = {
    "method": "traditional_ml",
    "model_type": "tfidf_logistic_regression_per_aspect",
    "aspects": ASPECTS,
    "labels": LABELS,
    "models": models,
}

joblib.dump(artifact, MODEL_PATH)
REPORT_PATH.write_text(json.dumps(report, indent=2), encoding="utf-8")

print("Saved model:", MODEL_PATH)
print("Saved report:", REPORT_PATH)
print("Model file size:", MODEL_PATH.stat().st_size, "bytes")
print("Report file size:", REPORT_PATH.stat().st_size, "bytes")

Saved model: /Users/kafe/Desktop/NLP_Final Project/absa_v01/backends/traditional_ml/model.joblib
Saved report: /Users/kafe/Desktop/NLP_Final Project/absa_v01/backends/traditional_ml/training_report.json
Model file size: 1299849 bytes
Report file size: 5773 bytes


## Step 9: Demo Prediction

This function mirrors `backends/traditional_ml/predict.py`. It returns the shared project output format.

In [9]:
def predict_review(review: str) -> dict:
    review = review.strip()
    if not review:
        raise ValueError("Review must not be empty.")

    results = []
    for aspect in ASPECTS:
        sentiment = models[aspect].predict([review])[0]
        results.append({"aspect": aspect, "sentiment": sentiment})

    return {
        "review": review,
        "method": "traditional_ml",
        "results": results,
    }

sample_review = "The food was delicious."
prediction = predict_review(sample_review)
prediction

{'review': 'The food was delicious.',
 'method': 'traditional_ml',
 'results': [{'aspect': 'Food', 'sentiment': 'Positive'},
  {'aspect': 'Service', 'sentiment': 'Unknown'},
  {'aspect': 'Price', 'sentiment': 'Unknown'},
  {'aspect': 'Eating Environment / Ambiance', 'sentiment': 'Unknown'}]}

## Notes for the Report

This model is a baseline. It is useful because it is fast, explainable, and measurable with F1-score. However, it can still make mistakes on mixed-sentiment sentences because TF-IDF does not deeply understand which opinion word belongs to which aspect. That limitation is one reason to compare it against the rule-based method and, later, a BERT model.